# Activity 1: RAGAS Evaluation with Cost Analysis

Evaluates two RAG pipelines head-to-head on the same cat-health-guide corpus:

| Arm | Embedding | Chat Model |
|-----|-----------|------------|
| **Fireworks** | `qwen3-embedding-8b` | `gpt-oss-20b` |
| **OpenAI** | `text-embedding-3-small` | `gpt-4.1-mini` |

**Metrics collected**
- RAGAS: `context_precision`, `context_recall`, `faithfulness`, `answer_correctness`
- Token usage: tracked per-call via `get_openai_callback` and mirrored in LangSmith
- Estimated cost at scale: per-query USD + projected 10k-query spend

Both pipelines are traced in separate LangSmith projects (`s10-rag-fireworks`, `s10-rag-openai`).

## 1. Setup

In [1]:
# ── Compatibility: stub the VertexAI path ragas still imports in 0.4.x ────────
import sys
from unittest.mock import MagicMock
sys.modules.setdefault("langchain_community.chat_models.vertexai", MagicMock())

import os
import time
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from dotenv import load_dotenv
load_dotenv(override=True)

print("Setup complete.")

Setup complete.


In [2]:
# ── Environment check ─────────────────────────────────────────────────────────
REQUIRED = {
    "FIREWORKS_API_KEY": "fireworks.ai/api-keys",
    "OPENAI_API_KEY":    "platform.openai.com/api-keys",
    "LANGSMITH_API_KEY": "smith.langchain.com (Settings > API Keys)",
}
missing = [k for k in REQUIRED if not os.environ.get(k)]
if missing:
    for k in missing:
        print(f"  ✗  {k}  →  get it at {REQUIRED[k]}")
    raise EnvironmentError(f"Add the missing keys to .env and restart the kernel.")
print("✅  All required environment variables present.")

# LangSmith tracing — ON for both runs
os.environ["LANGSMITH_TRACING"] = "true"
LANGSMITH_PROJECT_FW = "s10-rag-fireworks"
LANGSMITH_PROJECT_OA = "s10-rag-openai"
print(f"LangSmith projects: '{LANGSMITH_PROJECT_FW}'  |  '{LANGSMITH_PROJECT_OA}'")

✅  All required environment variables present.
LangSmith projects: 's10-rag-fireworks'  |  's10-rag-openai'


In [3]:
# ── Provider configuration ────────────────────────────────────────────────────
FIREWORKS_BASE_URL       = "https://api.fireworks.ai/inference/v1"
FIREWORKS_EMBED_MODEL    = os.environ.get(
    "FIREWORKS_EMBEDDING_MODEL", "accounts/fireworks/models/qwen3-embedding-8b"
)
FIREWORKS_CHAT_MODEL     = os.environ.get(
    "FIREWORKS_CHAT_MODEL", "accounts/fireworks/models/gpt-oss-20b"
)
OPENAI_EMBED_MODEL       = "text-embedding-3-small"
OPENAI_CHAT_MODEL        = "gpt-4.1-mini"

# Approximate pricing — verify at provider dashboards before quoting
# Fireworks serverless: https://fireworks.ai/pricing
# OpenAI: https://openai.com/api/pricing/
PRICING = {
    "fireworks": {"input_per_1m": 0.22,  "output_per_1m": 0.88,  "embed_per_1m": 0.016},
    "openai":    {"input_per_1m": 0.40,  "output_per_1m": 1.60,  "embed_per_1m": 0.020},
}

print(f"Fireworks  chat: {FIREWORKS_CHAT_MODEL}")
print(f"Fireworks embed: {FIREWORKS_EMBED_MODEL}")
print(f"OpenAI     chat: {OPENAI_CHAT_MODEL}")
print(f"OpenAI   embed: {OPENAI_EMBED_MODEL}")

Fireworks  chat: accounts/fireworks/models/gpt-oss-20b
Fireworks embed: accounts/fireworks/models/qwen3-embedding-8b
OpenAI     chat: gpt-4.1-mini
OpenAI   embed: text-embedding-3-small


## 2. Load & Split Documents

In [4]:
import tiktoken
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def _tiktoken_len(text: str) -> int:
    return len(tiktoken.encoding_for_model("gpt-4o").encode(text))

# Load — same glob as app/rag.py
loader = DirectoryLoader("data", glob="**/*.pdf", loader_cls=PyMuPDFLoader)
documents = loader.load()
print(f"Loaded {len(documents)} document page(s)")

# Split — identical params to app/rag.py for parity
splitter = RecursiveCharacterTextSplitter(
    chunk_size=750, chunk_overlap=0, length_function=_tiktoken_len
)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

Loaded 22 document page(s)
Created 42 chunks


## 3. RAG Pipeline Factory

Mirrors `_build_rag_graph` in `app/rag.py` without modifying the production file.
The same prompt is used for both arms to ensure apples-to-apples comparison.

In [6]:
from langchain_openai import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.callbacks import get_openai_callback

# Shared prompt — same as app/rag.py
_RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("human",
     "\n#CONTEXT:\n{context}\n\nQUERY:\n{query}\n\n"
     "Use the provided context to answer the query. "
     "Only use the provided context. "
     "If the answer is not in the context, respond with \"I don't know\".")
])


def build_rag_pipeline(
    *,
    provider: str,
    embedding_model: str,
    chat_model: str,
    api_key: str,
    api_base: str | None = None,
    embedding_dimensions: int | None = None,
    collection_name: str,
) -> dict:
    """Build an in-memory RAG pipeline (embeddings + vector store + generator)."""
    embed_kwargs: dict = dict(
        model=embedding_model,
        openai_api_key=api_key,
        check_embedding_ctx_length=False,
    )
    if api_base:
        embed_kwargs["openai_api_base"] = api_base
    if embedding_dimensions:
        embed_kwargs["dimensions"] = embedding_dimensions

    llm_kwargs: dict = dict(model=chat_model, temperature=0, openai_api_key=api_key)
    if api_base:
        llm_kwargs["openai_api_base"] = api_base

    embeddings = OpenAIEmbeddings(**embed_kwargs)
    llm        = ChatOpenAI(**llm_kwargs)

    print(f"  Building vector store for '{provider}' …")
    vectorstore = QdrantVectorStore.from_documents(
        documents=chunks,
        embedding=embeddings,
        location=":memory:",
        collection_name=collection_name,
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    chain     = _RAG_PROMPT | llm | StrOutputParser()

    print(f"  ✅  '{provider}' pipeline ready")
    return {"provider": provider, "retriever": retriever, "chain": chain}


def run_rag(pipeline: dict, question: str) -> dict:
    """Run one question; returns response, contexts, and token usage."""
    retrieved_docs = pipeline["retriever"].invoke(question)
    contexts       = [doc.page_content for doc in retrieved_docs]
    with get_openai_callback() as cb:
        response = pipeline["chain"].invoke({"context": retrieved_docs, "query": question})
    return {
        "response":          response,
        "contexts":          contexts,
        "prompt_tokens":     cb.prompt_tokens,
        "completion_tokens": cb.completion_tokens,
        "total_tokens":      cb.total_tokens,
    }

In [7]:
print("Building pipelines (embedding all chunks — may take ~1-2 min for Fireworks)…")

fw_pipeline = build_rag_pipeline(
    provider="fireworks",
    embedding_model=FIREWORKS_EMBED_MODEL,
    chat_model=FIREWORKS_CHAT_MODEL,
    api_key=os.environ["FIREWORKS_API_KEY"],
    api_base=FIREWORKS_BASE_URL,
    embedding_dimensions=4096,
    collection_name="rag_fireworks",
)

oa_pipeline = build_rag_pipeline(
    provider="openai",
    embedding_model=OPENAI_EMBED_MODEL,
    chat_model=OPENAI_CHAT_MODEL,
    api_key=os.environ["OPENAI_API_KEY"],
    api_base=None,
    embedding_dimensions=None,
    collection_name="rag_openai",
)

print("\nBoth pipelines ready.")

Building pipelines (embedding all chunks — may take ~1-2 min for Fireworks)…
  Building vector store for 'fireworks' …
  ✅  'fireworks' pipeline ready
  Building vector store for 'openai' …
  ✅  'openai' pipeline ready

Both pipelines ready.


## 4. Test Set

10 domain-specific questions covering the main topics in the cat health guide
(vaccinations, wellness, dental, nutrition, parasites, pain, spay/neuter, seniors, obesity, heartworm).
Reference answers are factual baselines used by the RAGAS judge.

In [8]:
TESTSET = [
    {
        "question": "What core vaccines are recommended for all cats?",
        "reference": (
            "Core vaccines for cats include feline panleukopenia (FPV), feline herpesvirus "
            "(FHV-1), and feline calicivirus (FCV) — often given as the combination FVRCP "
            "vaccine — plus rabies. These are recommended regardless of lifestyle."
        ),
    },
    {
        "question": "How often should adult cats have veterinary wellness exams?",
        "reference": (
            "Adult cats should have a wellness exam at least once per year. Senior cats "
            "aged 7 and older benefit from biannual (twice-yearly) check-ups to detect "
            "age-related changes early."
        ),
    },
    {
        "question": "What are the common signs of dental disease in cats?",
        "reference": (
            "Common signs include bad breath, visible tartar or calculus, red or swollen "
            "gums, drooling, pawing at the mouth, difficulty eating, and dropping food."
        ),
    },
    {
        "question": "How does kitten nutrition differ from adult cat nutrition?",
        "reference": (
            "Kitten food has higher protein, fat, and calorie density to fuel rapid growth. "
            "It also supplies more calcium and phosphorus for bone development. Kittens "
            "should transition to adult food around 12 months of age."
        ),
    },
    {
        "question": "What parasite prevention is recommended for indoor-only cats?",
        "reference": (
            "Even indoor cats can be exposed to fleas, ear mites, and intestinal parasites. "
            "Year-round flea prevention, annual fecal exams, and heartworm preventives (in "
            "endemic regions) are recommended for indoor cats."
        ),
    },
    {
        "question": "What behavioral signs indicate a cat may be in pain?",
        "reference": (
            "Behavioral pain signs in cats include hiding, reduced activity, grooming changes "
            "(neglect or over-grooming), decreased appetite, reluctance to jump, vocalization, "
            "aggression when touched, and facial changes such as narrowed eyes and flattened ears."
        ),
    },
    {
        "question": "At what age should kittens typically be spayed or neutered?",
        "reference": (
            "Kittens are generally recommended to be spayed or neutered between 4 and 6 months "
            "of age, before the first heat cycle in females. Early spay/neuter at 8 weeks is "
            "practiced in shelter settings."
        ),
    },
    {
        "question": "How should the diet of senior cats differ from that of adult cats?",
        "reference": (
            "Senior cats (7+ years) often benefit from highly digestible protein to maintain "
            "muscle mass, adjusted calorie levels, joint-support supplements, and increased "
            "dietary moisture via wet food to support kidney function."
        ),
    },
    {
        "question": "What health risks are associated with feline obesity?",
        "reference": (
            "Obese cats face elevated risks of diabetes mellitus, hepatic lipidosis, "
            "osteoarthritis, urinary tract disease, and reduced life expectancy. Weight "
            "management includes measured portions, fewer treats, and increased play activity."
        ),
    },
    {
        "question": "Why is heartworm prevention important for cats?",
        "reference": (
            "There is no approved treatment for feline heartworm disease, making prevention "
            "critical. Monthly preventives are recommended even for indoor cats in endemic "
            "areas because mosquitoes — the transmission vector — can enter homes."
        ),
    },
]

print(f"Test set: {len(TESTSET)} questions ready.")

Test set: 10 questions ready.


## 5. Run Both Pipelines with LangSmith Tracing

Each arm is run under a separate LangSmith project so token usage, latency, and cost
can be compared in the dashboard:
- 🔗 [s10-rag-fireworks](https://smith.langchain.com) 
- 🔗 [s10-rag-openai](https://smith.langchain.com)

In [9]:
def run_all(pipeline: dict, testset: list[dict], langsmith_project: str) -> list[dict]:
    """Run the full testset through one pipeline; traces go to langsmith_project."""
    os.environ["LANGSMITH_PROJECT"] = langsmith_project
    provider = pipeline["provider"]
    results  = []
    print(f"\n▶ Running '{provider}' (LangSmith: {langsmith_project})")
    for i, item in enumerate(testset, 1):
        q = item["question"]
        print(f"  Q{i:02d}: {q[:70]}…")
        out = run_rag(pipeline, q)
        results.append({
            "question":          q,
            "reference":         item["reference"],
            "response":          out["response"],
            "contexts":          out["contexts"],
            "prompt_tokens":     out["prompt_tokens"],
            "completion_tokens": out["completion_tokens"],
            "total_tokens":      out["total_tokens"],
        })
        time.sleep(0.3)  # gentle rate-limit buffer
    total_tok = sum(r["total_tokens"] for r in results)
    print(f"  ✅  Done — {len(results)} answers, {total_tok:,} total tokens used")
    return results


fw_results = run_all(fw_pipeline, TESTSET, LANGSMITH_PROJECT_FW)
oa_results = run_all(oa_pipeline, TESTSET, LANGSMITH_PROJECT_OA)


▶ Running 'fireworks' (LangSmith: s10-rag-fireworks)
  Q01: What core vaccines are recommended for all cats?…
  Q02: How often should adult cats have veterinary wellness exams?…
  Q03: What are the common signs of dental disease in cats?…
  Q04: How does kitten nutrition differ from adult cat nutrition?…
  Q05: What parasite prevention is recommended for indoor-only cats?…
  Q06: What behavioral signs indicate a cat may be in pain?…
  Q07: At what age should kittens typically be spayed or neutered?…
  Q08: How should the diet of senior cats differ from that of adult cats?…
  Q09: What health risks are associated with feline obesity?…
  Q10: Why is heartworm prevention important for cats?…
  ✅  Done — 10 answers, 73,692 total tokens used

▶ Running 'openai' (LangSmith: s10-rag-openai)
  Q01: What core vaccines are recommended for all cats?…
  Q02: How often should adult cats have veterinary wellness exams?…
  Q03: What are the common signs of dental disease in cats?…
  Q04: How does ki

## 6. RAGAS Evaluation

Both arms are judged by the **same** `gpt-4.1-mini` judge to avoid provider bias.

| Metric | Measures |
|--------|----------|
| `context_precision` | Signal-to-noise in retrieved chunks |
| `context_recall` | Coverage of the reference answer by retrieved chunks |
| `faithfulness` | Factual groundedness of the response in the retrieved context |
| `answer_correctness` | Semantic similarity to the reference answer |

In [10]:
from ragas import EvaluationDataset

# SingleTurnSample — try both import locations across ragas 0.4.x
try:
    from ragas.dataset_schema import SingleTurnSample
except ImportError:
    from ragas import SingleTurnSample  # older fallback


def build_dataset(results: list[dict]) -> EvaluationDataset:
    samples = [
        SingleTurnSample(
            user_input=r["question"],
            retrieved_contexts=r["contexts"],
            response=r["response"],
            reference=r["reference"],
        )
        for r in results
    ]
    return EvaluationDataset(samples=samples)


fw_dataset = build_dataset(fw_results)
oa_dataset = build_dataset(oa_results)
print(f"Fireworks dataset: {len(fw_dataset)} samples")
print(f"OpenAI dataset   : {len(oa_dataset)} samples")

Fireworks dataset: 10 samples
OpenAI dataset   : 10 samples


In [11]:
from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_correctness,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI as _OAChat
from langchain_openai.embeddings import OpenAIEmbeddings as _OAEmbed

# Consistent judge for both arms — gpt-4.1-mini via OpenAI
judge_llm = LangchainLLMWrapper(_OAChat(
    model="gpt-4.1-mini",
    openai_api_key=os.environ["OPENAI_API_KEY"],
))
judge_embed = LangchainEmbeddingsWrapper(_OAEmbed(
    model="text-embedding-3-small",
    openai_api_key=os.environ["OPENAI_API_KEY"],
))

# Legacy ragas.metrics singletons are Metric instances compatible with evaluate()
METRICS = [context_precision, context_recall, faithfulness, answer_correctness]

# Route RAGAS judge calls to their own LangSmith project
os.environ["LANGSMITH_PROJECT"] = "s10-ragas-eval"

print("Evaluating Fireworks pipeline...")
fw_eval = evaluate(dataset=fw_dataset, metrics=METRICS, llm=judge_llm, embeddings=judge_embed)

print("Evaluating OpenAI pipeline...")
oa_eval = evaluate(dataset=oa_dataset, metrics=METRICS, llm=judge_llm, embeddings=judge_embed)

print("\nRAGAS evaluation complete.")

Evaluating Fireworks pipeline...


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Evaluating OpenAI pipeline...


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]


RAGAS evaluation complete.


In [13]:
import pandas as pd

METRIC_KEYS = ["context_precision", "context_recall", "faithfulness", "answer_correctness"]

# EvaluationResult indexing returns per-sample lists in this ragas version;
# aggregate via the mean of each metric column in the results DataFrame.
fw_df = fw_eval.to_pandas()
oa_df = oa_eval.to_pandas()

fw_scores = {m: round(float(fw_df[m].mean()), 4) for m in METRIC_KEYS}
oa_scores = {m: round(float(oa_df[m].mean()), 4) for m in METRIC_KEYS}

metrics_df = pd.DataFrame({
    "Metric":                          METRIC_KEYS,
    f"Fireworks ({FIREWORKS_CHAT_MODEL.split('/')[-1]})": [fw_scores[m] for m in METRIC_KEYS],
    f"OpenAI ({OPENAI_CHAT_MODEL})": [oa_scores[m] for m in METRIC_KEYS],
    "Δ (FW − OAI)":                   [round(fw_scores[m] - oa_scores[m], 4) for m in METRIC_KEYS],
}).set_index("Metric")

print("\n=== RAGAS Metrics Comparison ===")
print(metrics_df.to_string())


=== RAGAS Metrics Comparison ===
                    Fireworks (gpt-oss-20b)  OpenAI (gpt-4.1-mini)  Δ (FW − OAI)
Metric                                                                          
context_precision                    0.7083                 0.7250       -0.0167
context_recall                       0.5833                 0.5833        0.0000
faithfulness                         0.6526                 0.8608       -0.2082
answer_correctness                   0.3905                 0.3346        0.0559


## 7. Cost Analysis

Token counts are read directly from the `get_openai_callback` captured during the
inference runs above.  Cost is computed against published serverless pricing.

> **Note:** For the LangSmith visual cost dashboard, set a custom model price under
> *Settings → Model Pricing* for the Fireworks model IDs — LangSmith auto-prices OpenAI
> models but requires manual entries for third-party providers.

In [14]:
def aggregate_tokens(results: list[dict]) -> dict:
    return {
        "prompt_tokens":     sum(r["prompt_tokens"]     for r in results),
        "completion_tokens": sum(r["completion_tokens"] for r in results),
        "total_tokens":      sum(r["total_tokens"]      for r in results),
    }

fw_tok = aggregate_tokens(fw_results)
oa_tok = aggregate_tokens(oa_results)

print("Token usage over the test set:")
print(f"  Fireworks: {fw_tok}")
print(f"  OpenAI:    {oa_tok}")

Token usage over the test set:
  Fireworks: {'prompt_tokens': 36767, 'completion_tokens': 36925, 'total_tokens': 73692}
  OpenAI:    {'prompt_tokens': 36995, 'completion_tokens': 1523, 'total_tokens': 38518}


In [15]:
def compute_cost(tok: dict, pricing: dict, n_queries: int) -> dict:
    """Return cost breakdown given token counts and per-1M pricing."""
    input_usd  = (tok["prompt_tokens"]     / 1_000_000) * pricing["input_per_1m"]
    output_usd = (tok["completion_tokens"] / 1_000_000) * pricing["output_per_1m"]
    total_usd  = input_usd + output_usd
    cpq        = total_usd / n_queries if n_queries else 0
    return {
        "input_usd":            round(input_usd, 6),
        "output_usd":           round(output_usd, 6),
        "total_usd":            round(total_usd, 6),
        "cost_per_query_usd":   round(cpq, 7),
        "projected_10k_usd":    round(cpq * 10_000, 4),
        "projected_100k_usd":   round(cpq * 100_000, 3),
    }

n = len(TESTSET)
fw_cost = compute_cost(fw_tok, PRICING["fireworks"], n)
oa_cost = compute_cost(oa_tok, PRICING["openai"],    n)

ROWS = [
    f"Fireworks ({FIREWORKS_CHAT_MODEL.split('/')[-1]})",
    f"OpenAI ({OPENAI_CHAT_MODEL})",
    "Savings (OAI − FW)",
]

def _diff(a, b): return round(b - a, 6)

cost_df = pd.DataFrame({
    "Provider":                ROWS,
    "Prompt Tokens":           [fw_tok["prompt_tokens"],        oa_tok["prompt_tokens"],        "—"],
    "Completion Tokens":       [fw_tok["completion_tokens"],    oa_tok["completion_tokens"],    "—"],
    "Test-set Cost ($)": [
        f"${fw_cost['total_usd']:.5f}",
        f"${oa_cost['total_usd']:.5f}",
        f"${_diff(fw_cost['total_usd'], oa_cost['total_usd']):.5f}",
    ],
    "Cost / Query ($)": [
        f"${fw_cost['cost_per_query_usd']:.7f}",
        f"${oa_cost['cost_per_query_usd']:.7f}",
        f"${_diff(fw_cost['cost_per_query_usd'], oa_cost['cost_per_query_usd']):.7f}",
    ],
    "Proj. 10k Queries ($)": [
        f"${fw_cost['projected_10k_usd']:.2f}",
        f"${oa_cost['projected_10k_usd']:.2f}",
        f"${_diff(fw_cost['projected_10k_usd'], oa_cost['projected_10k_usd']):.2f}",
    ],
    "Proj. 100k Queries ($)": [
        f"${fw_cost['projected_100k_usd']:.2f}",
        f"${oa_cost['projected_100k_usd']:.2f}",
        f"${_diff(fw_cost['projected_100k_usd'], oa_cost['projected_100k_usd']):.2f}",
    ],
}).set_index("Provider")

print("\n=== Cost Comparison (chat tokens only; embeddings excluded) ===")
print(cost_df.to_string())
print(f"\nPricing note: Fireworks rates @ $0.22/$0.88 per 1M in/out; "
      f"OpenAI rates @ $0.40/$1.60 per 1M in/out (verify current rates before quoting).")


=== Cost Comparison (chat tokens only; embeddings excluded) ===
                        Prompt Tokens Completion Tokens Test-set Cost ($) Cost / Query ($) Proj. 10k Queries ($) Proj. 100k Queries ($)
Provider                                                                                                                               
Fireworks (gpt-oss-20b)         36767             36925          $0.04058       $0.0040583                $40.58                $405.83
OpenAI (gpt-4.1-mini)           36995              1523          $0.01724       $0.0017235                $17.23                $172.35
Savings (OAI − FW)                  —                 —         $-0.02335      $-0.0023350               $-23.35               $-233.48

Pricing note: Fireworks rates @ $0.22/$0.88 per 1M in/out; OpenAI rates @ $0.40/$1.60 per 1M in/out (verify current rates before quoting).


## 8. Per-Query Breakdown

In [16]:
per_query_rows = []
for i, (fw, oa, item) in enumerate(zip(fw_results, oa_results, TESTSET), 1):
    per_query_rows.append({
        "Q#":      i,
        "Question": item["question"][:55] + "…",
        "FW tokens": fw["total_tokens"],
        "OAI tokens": oa["total_tokens"],
        "FW response (preview)": fw["response"][:80] + "…",
        "OAI response (preview)": oa["response"][:80] + "…",
    })

pq_df = pd.DataFrame(per_query_rows).set_index("Q#")
pd.set_option("display.max_colwidth", 90)
display(pq_df)

,Question,FW tokens,OAI tokens,FW response (preview),OAI response (preview)
Q#,,,,,
1,What core vaccines are recommended for all cats?…,3337,3891,The core vaccines recommended for all cats are:\n\n- Rabies virus \n- Feline herpe…,"The core vaccines recommended for all cats are rabies virus, feline herpesvirus …"
2,How often should adult cats have veterinary wellness ex…,3909,3478,Adult cats should have a veterinary wellness exam **at least once a year** (with…,Adult cats should have a minimum of annual veterinary wellness examinations. For…
3,What are the common signs of dental disease in cats?…,4460,3958,I don't know.…,Common signs of dental disease in cats include the presence of dental pathology …
4,How does kitten nutrition differ from adult cat nutriti…,4929,4018,"Kittens need a diet that supports rapid growth, so their food is formulated to b…",Kitten nutrition differs from adult cat nutrition primarily in energy requiremen…
5,What parasite prevention is recommended for indoor-only…,3961,3877,"For indoor‑only cats, the guideline recommends routine use of a **broad‑spectrum…","For indoor-only cats, routine, regular use of broad-spectrum parasite prevention…"
6,What behavioral signs indicate a cat may be in pain?…,4610,3853,Behavioral clues that a cat may be experiencing pain include:\n\n* **Changes in gr…,Behavioral signs that indicate a cat may be in pain include changes in grooming …
7,At what age should kittens typically be spayed or neute…,3856,3576,I don't know.…,I don't know.…
8,How should the diet of senior cats differ from that of …,36786,4156,…,The diet of senior cats (greater than 10 years of age) should differ from that o…
9,What health risks are associated with feline obesity?…,3890,3845,"Feline obesity can predispose cats to several chronic health problems, including…","Health risks associated with feline obesity, as described in the provided contex…"


## 9. Analysis & Observations

Results below are populated from this notebook run.

### Quality (RAGAS)

| Dimension | Fireworks | OpenAI | Winner |
|-----------|-----------|--------|--------|
| Context precision (retrieval signal-to-noise) | 0.7083 | 0.7250 | OpenAI |
| Context recall (coverage) | 0.5833 | 0.5833 | Tie |
| Faithfulness (grounded in context) | 0.6526 | 0.8608 | OpenAI |
| Answer correctness (vs reference) | 0.3905 | 0.3346 | Fireworks |

**Key takeaway:** OpenAI performed better on grounding-focused metrics (context precision and faithfulness), while Fireworks produced slightly higher answer correctness. Context recall was identical, indicating retrieval coverage was similar and most differences came from generation quality/grounding behavior.

---

### Cost

| Provider | Cost / query | Proj. 10k queries | Proj. 100k queries |
|----------|-------------|-------------------|--------------------|
| Fireworks gpt-oss-20b | $0.0040583 | $40.58 | $405.83 |
| OpenAI gpt-4.1-mini   | $0.0017235 | $17.23 | $172.35 |
| **Savings (using OpenAI)** | **$0.0023350** | **$23.35** | **$233.48** |

**Key takeaway:** For this run and the pricing assumptions in this notebook, OpenAI was materially cheaper because completion token usage was much lower (1,523 vs 36,925). Cost sensitivity here is dominated by output token volume, not just list price per 1M tokens.

---

### LangSmith Tracing

- Fireworks traces: https://smith.langchain.com -> project **s10-rag-fireworks**
- OpenAI traces: https://smith.langchain.com -> project **s10-rag-openai**
- RAGAS judge: https://smith.langchain.com -> project **s10-ragas-eval**

Screenshots/screen-share of per-run token counts and latency in LangSmith are recommended for the Loom video.

---

### Summary

In this experiment, OpenAI gpt-4.1-mini achieved stronger groundedness (faithfulness + context precision) and substantially lower cost, while Fireworks gpt-oss-20b slightly led on answer correctness. If production priorities are reliability and cost, OpenAI is the better default from this run; if answer-style alignment matters more, Fireworks remains competitive and may be improved with prompt and decoding tuning.